# 🎵 Download YouTube Audio to Google Drive

This notebook downloads audio-only files from YouTube videos stored in the `bronze_events_youtube` database table and saves them to Google Drive organized by channel and date.

**Features:**
- 🎵 Audio-only downloads (opus format, ~10-20 MB/hour)
- 📁 Organized by channel → `YYYY-MM-DD_title.opus`
- ⏭️ Skips already downloaded files
- 📊 Progress tracking and resumable
- ☁️ Works seamlessly with Google Drive

**Output Structure:**
```
youtube_audio/
├── City-of-Seattle_UCxxx/
│   ├── 2026-05-01_City Council Meeting.opus
│   └── 2026-04-28_Planning Commission.opus
├── Portland-City-Council_UCyyy/
│   └── 2026-05-02_Council Session.opus
```

## Step 1: Mount Google Drive

Mount your Google Drive to access the project files and store downloaded audio.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Navigate to Project

Change directory to the Open Navigator project in your Google Drive.

In [2]:
%cd /content/drive/MyDrive/CommunityOne/open-navigator

# Pull latest code
!git config core.hooksPath /dev/null
!git pull origin main

/content/drive/MyDrive/CommunityOne/open-navigator
From https://github.com/getcommunityone/open-navigator
 * branch            main       -> FETCH_HEAD
Already up to date.


## Step 3: Install Dependencies

Install required Python packages for downloading audio from YouTube.

In [3]:
!pip install -q yt-dlp loguru psycopg2-binary python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 52.8 MB/s eta 0:00:0000:0100:01


## Step 4: Set Environment Variables

Configure **Neon cloud database** connection using Colab Secrets.

⚠️ **Important**: This notebook connects to **Neon (cloud)**, not localhost!

### Set up Colab Secret:
1. Click the 🔑 key icon in left sidebar
2. Add secret named: `NEON_DATABASE_URL`
3. Paste your Neon connection string
4. Toggle "Notebook access" ON

In [4]:
import os
from google.colab import userdata

# Get Neon cloud database URL from Colab secret
# Secret name: NEON_DATABASE_URL (what you set in Colab Secrets panel)
# Sets env var: NEON_DATABASE_URL_DEV (what the script expects)
neon_url = userdata.get('NEON_DATABASE_URL')
os.environ['NEON_DATABASE_URL_DEV'] = neon_url

print(f"✅ Database configured: Neon cloud PostgreSQL")
print(f"   Connected to: {neon_url.split('@')[1].split('/')[0] if '@' in neon_url else 'unknown'}")

### ✅ Verify Database Connection (Optional)

Run this cell to test your database connection before downloading files.

## Step 5: Create Output Directory

Create the directory where audio files will be stored.

In [5]:
!mkdir -p /content/drive/MyDrive/CommunityOne/youtube_audio

## Step 6: Download Audio - Recent Videos (Last 30 Days)

Download a sample of recent videos. Good for testing before downloading larger batches.

In [6]:
!python scripts/datasources/youtube/download_audio_to_drive.py \
  --output-dir /content/drive/MyDrive/CommunityOne/youtube_audio \
  --days 30 \
  --limit 50

2026-05-06 15:02:24.926 | INFO     | __main__:run:236 - ================================================================================
2026-05-06 15:02:24.926 | INFO     | __main__:run:237 - YOUTUBE AUDIO DOWNLOADER
2026-05-06 15:02:24.926 | INFO     | __main__:run:238 - ================================================================================
2026-05-06 15:02:24.926 | INFO     | __main__:run:239 - Output directory: /content/drive/MyDrive/CommunityOne/youtube_audio
2026-05-06 15:02:24.927 | INFO     | __main__:run:240 - Database: host:5432/open_navigator
2026-05-06 15:02:24.927 | INFO     | __main__:run:243 - Limit: 50 videos
2026-05-06 15:02:24.927 | INFO     | __main__:run:249 - 
2026-05-06 15:02:24.927 | INFO     | __main__:run:252 - 📊 Querying database for videos...
Traceback (most recent call last):
  File "/content/drive/MyDrive/CommunityOne/open-navigator/scripts/datasources/youtube/download_audio_to_drive.py", line 400, in <module>
    main()
  File "/content/drive/MyD

## Step 7: Download Audio - Specific States

Filter downloads by state codes. Useful for focusing on particular regions.

**Example:** Alabama (AL), Massachusetts (MA), Wisconsin (WI)

In [7]:
!python scripts/datasources/youtube/download_audio_to_drive.py \
  --output-dir /content/drive/MyDrive/CommunityOne/youtube_audio \
  --states AL,MA,WI \
  --limit 100

2026-05-06 15:02:26.284 | INFO     | __main__:run:236 - ================================================================================
2026-05-06 15:02:26.284 | INFO     | __main__:run:237 - YOUTUBE AUDIO DOWNLOADER
2026-05-06 15:02:26.285 | INFO     | __main__:run:238 - ================================================================================
2026-05-06 15:02:26.285 | INFO     | __main__:run:239 - Output directory: /content/drive/MyDrive/CommunityOne/youtube_audio
2026-05-06 15:02:26.285 | INFO     | __main__:run:240 - Database: host:5432/open_navigator
2026-05-06 15:02:26.285 | INFO     | __main__:run:243 - Limit: 100 videos
2026-05-06 15:02:26.285 | INFO     | __main__:run:247 - States filter: AL, MA, WI
2026-05-06 15:02:26.285 | INFO     | __main__:run:249 - 
2026-05-06 15:02:26.285 | INFO     | __main__:run:252 - 📊 Querying database for videos...
Traceback (most recent call last):
  File "/content/drive/MyDrive/CommunityOne/open-navigator/scripts/datasources/youtube/downl

## Step 8: Download Audio - Specific Channels

Filter downloads by channel names (partial match, case-insensitive).

In [8]:
!python scripts/datasources/youtube/download_audio_to_drive.py \
  --output-dir /content/drive/MyDrive/CommunityOne/youtube_audio \
  --channels "Seattle,Portland,Boston" \
  --limit 200

2026-05-06 15:02:27.312 | INFO     | __main__:run:236 - ================================================================================
2026-05-06 15:02:27.312 | INFO     | __main__:run:237 - YOUTUBE AUDIO DOWNLOADER
2026-05-06 15:02:27.312 | INFO     | __main__:run:238 - ================================================================================
2026-05-06 15:02:27.312 | INFO     | __main__:run:239 - Output directory: /content/drive/MyDrive/CommunityOne/youtube_audio
2026-05-06 15:02:27.312 | INFO     | __main__:run:240 - Database: host:5432/open_navigator
2026-05-06 15:02:27.312 | INFO     | __main__:run:243 - Limit: 200 videos
2026-05-06 15:02:27.312 | INFO     | __main__:run:245 - Channels filter: Seattle, Portland, Boston
2026-05-06 15:02:27.312 | INFO     | __main__:run:249 - 
2026-05-06 15:02:27.312 | INFO     | __main__:run:252 - 📊 Querying database for videos...
Traceback (most recent call last):
  File "/content/drive/MyDrive/CommunityOne/open-navigator/scripts/datasour

## Step 9: Check Downloaded Files

List downloaded files and check the output structure.

In [9]:
# List audio directory
!ls -lh /content/drive/MyDrive/CommunityOne/youtube_audio/

# Count total opus files
!find /content/drive/MyDrive/CommunityOne/youtube_audio -name "*.opus" | wc -l

# List channel directories
!ls -d /content/drive/MyDrive/CommunityOne/youtube_audio/*/

total 0
0
ls: cannot access '/content/drive/MyDrive/CommunityOne/youtube_audio/*/': No such file or directory


## Step 10 (Optional): Download All 2026 Videos

⚠️ **WARNING:** This will download thousands of files and may take hours!

**Considerations:**
- Google Drive free tier: 15 GB
- Typical meeting: 15-30 MB (1 hour)
- Estimated capacity: ~500-1000 meetings

**Uncomment the cell below to run.**

In [10]:
# !python scripts/datasources/youtube/download_audio_to_drive.py \
#   --output-dir /content/drive/MyDrive/CommunityOne/youtube_audio \
#   --days 180 \
#   --limit 1000

## 📝 Command Line Options

Available flags for `download_audio_to_drive.py`:

| Option | Description | Example |
|--------|-------------|---------|
| `--output-dir PATH` | Output directory | `/content/drive/MyDrive/youtube_audio` |
| `--limit N` | Max videos to download | `--limit 100` |
| `--channels "A,B"` | Filter by channel names (partial match) | `--channels "Seattle,Portland"` |
| `--states "AL,MA"` | Filter by state codes | `--states AL,MA,WI` |
| `--days N` | Only videos from last N days | `--days 30` |
| `--no-skip-existing` | Re-download existing files | `--no-skip-existing` |
| `--database-url URL` | Database connection string | See Step 4 |

## 🎵 Audio Format

- **Codec:** Opus (best quality/size ratio)
- **Size:** ~10-20 MB per hour of audio
- **Quality:** 128 kbps
- **Container:** .opus file

## 💡 Tips

- ✅ Files are automatically skipped if already downloaded
- ✅ Use `--limit` to test with small batches first
- ✅ Monitor Google Drive storage quota (15 GB free tier)
- ✅ Download will continue from where it left off if interrupted
- ✅ Use Colab secrets to store database credentials securely

## 🔗 Next Steps

After downloading audio files:

1. **Transcribe with Whisper** (OpenAI or local model)
2. **Analyze with Gemini AI** for decisions/topics
3. **Store transcripts** in `bronze.bronze_transcripts_raw` table
4. **Extract entities** with dbt models

See the [Google Colab Setup Guide](../../../website/docs/guides/google-colab-setup.md) for the complete workflow.